# Step 03 — PCA & Clustering

This notebook explores the PCA dimensionality reduction and k-means clustering
results produced by `pipelines/03_cluster.py`.

Two cluster assignments are always produced:
- `cluster`: best k chosen by silhouette score (capped at `k_cap`, default 5)
- `balanced_cluster`: size-constrained k-means at a fixed `balanced_k` (default 5)

Downstream regime labelling uses `balanced_cluster` by default because
equal-size clusters give better per-regime statistics with limited data.

**Run `python pipelines/03_cluster.py` before executing this notebook.**

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR
from market_regime import plotting

setup_logging("INFO")
log = logging.getLogger("03_clustering")

cfg = load()

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("RunConfig:", run_cfg)
print("DATA_DIR: ", DATA_DIR)

## Load Clustering Artifacts

Three parquet files are loaded from `data/regimes/`:
- `cluster_labels.parquet` — quarter → cluster assignment (both `cluster` and `balanced_cluster`)
- `pca_components.parquet` — the 5 PCA components for each quarter
- `kmeans_scores.parquet` — silhouette / Calinski-Harabasz / Davies-Bouldin scores for each k

In [ ]:
REGIMES_DIR = DATA_DIR / "regimes"

labels = None
pca_df = None
scores = None

try:
    labels = pd.read_parquet(REGIMES_DIR / "cluster_labels.parquet")
    print(f"Labels loaded:         {labels.shape[0]} quarters, columns: {list(labels.columns)}")
except FileNotFoundError:
    print("ERROR: cluster_labels.parquet not found. Run pipelines/03_cluster.py first.")

try:
    pca_df = pd.read_parquet(REGIMES_DIR / "pca_components.parquet")
    print(f"PCA components loaded: {pca_df.shape[0]} quarters x {pca_df.shape[1]} components")
except FileNotFoundError:
    print("ERROR: pca_components.parquet not found. Run pipelines/03_cluster.py first.")

try:
    scores = pd.read_parquet(REGIMES_DIR / "kmeans_scores.parquet")
    print(f"K-sweep scores loaded: {scores.shape[0]} rows, columns: {list(scores.columns)}")
except FileNotFoundError:
    print("ERROR: kmeans_scores.parquet not found. Run pipelines/03_cluster.py first.")

## PCA Explained Variance

The pipeline uses a fixed 5 PCA components (per `n_pca_components` in settings.yaml).
Explained variance is reported if stored separately.

In [ ]:
if pca_df is not None:
    print(f"PCA columns: {list(pca_df.columns)}")
    print(f"Shape:       {pca_df.shape}")
    print()
    # Check for separately-stored explained variance
    ev_path = REGIMES_DIR / "pca_explained_variance.parquet"
    try:
        ev = pd.read_parquet(ev_path)
        print("PCA explained variance:")
        display(ev)
    except FileNotFoundError:
        print("(pca_explained_variance.parquet not found — variance not stored separately)")
    print()
    print("First 5 rows of PCA components:")
    display(pca_df.head())

## K-Sweep Elbow Curves

Three metrics are plotted across the range of k values tested (default k=2..12).
The vertical dashed line marks the automatically chosen best k (max silhouette).

In [ ]:
if scores is not None:
    print("K-sweep scores:")
    display(scores)

    if "silhouette" in scores.columns and "k" in scores.columns:
        best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
        print(f"\nBest k (max silhouette): {best_k}")
        plotting.plot_elbow_curve(scores, best_k, run_cfg)
    else:
        print(f"Scores DataFrame missing 'silhouette' or 'k' columns.")
        print(f"Available columns: {list(scores.columns)}")

## PCA Scatter Plot

Two scatter panels: PC1 vs PC2, and PC3 vs PC4.
Points are coloured by `balanced_cluster` assignment.
Well-separated clusters in PC space indicate a clean macro regime structure.

In [ ]:
if pca_df is not None and labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_pca_scatter(pca_df, labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_pca_scatter(pca_df, labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available columns: {list(labels.columns)}")

## Cluster Sizes

Bar chart showing how many quarters fall into each cluster.
The balanced clustering constrains cluster sizes to be within ±2 of the bucket size.

In [ ]:
if labels is not None:
    if "balanced_cluster" in labels.columns:
        plotting.plot_cluster_sizes(labels["balanced_cluster"], {}, run_cfg)
    elif "cluster" in labels.columns:
        print("'balanced_cluster' not found — falling back to 'cluster'")
        plotting.plot_cluster_sizes(labels["cluster"], {}, run_cfg)
    else:
        print(f"No cluster column found. Available columns: {list(labels.columns)}")

## Cluster Value Counts

Exact quarter counts for both cluster assignment strategies.

In [ ]:
if labels is not None:
    if "cluster" in labels.columns:
        print("=== cluster (silhouette-selected k) ===")
        print(labels["cluster"].value_counts().sort_index().to_string())
        print()

    if "balanced_cluster" in labels.columns:
        print("=== balanced_cluster (size-constrained k) ===")
        print(labels["balanced_cluster"].value_counts().sort_index().to_string())
        print()

    print("=== Sample rows ===")
    display(labels.head(10))